In [0]:
# Configuration
SOURCE_PATH = "/Volumes/agentbricks/sector_data_raw/sector_data_volumes"
TARGET_SCHEMA = "agentbricks.sector_data_bronze"

# Mapping: source CSV filename -> target table name
TABLES = {
    "car_fleet_cz.csv": "car_fleet_cz",
    "new_registrations_cz.csv": "new_registrations_cz",
    "new_registrations_fuels_cz.csv": "new_registrations_fuels_cz",
    "mpo.csv": "mpo_industry_data",
}

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import os

def ingest_csv_to_bronze(source_path: str, table_name: str, full_table: str):
    """Read a CSV file and write it as a Delta table in the bronze layer."""
    print(f"\nIngesting: {source_path}")
    print(f"  -> Target: {full_table}")
    
    # Read CSV with header and schema inference
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(source_path)
    )
    
    # Add ingestion metadata
    df = (df
        .withColumn("_source_file", F.lit(os.path.basename(source_path)))
        .withColumn("_ingested_at", F.lit(datetime.now().isoformat()).cast("timestamp"))
    )
    
    # Write as Delta table (overwrite for full refresh of these small reference files)
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table)
    
    row_count = spark.table(full_table).count()
    print(f"  ✓ Written {row_count} rows, {len(df.columns)} columns")
    return row_count

print("Ingestion function defined.")

In [0]:
# Ingest all CSV files into bronze Delta tables
results = {}

for csv_file, table_name in TABLES.items():
    source_file = os.path.join(SOURCE_PATH, csv_file)
    full_table = f"{TARGET_SCHEMA}.{table_name}"
    
    if not os.path.exists(source_file):
        print(f"⚠️  File not found: {source_file}")
        continue
    
    row_count = ingest_csv_to_bronze(source_file, table_name, full_table)
    results[table_name] = row_count

print(f"\n{'='*60}")
print("INGESTION COMPLETE")
print(f"{'='*60}")
for table, rows in results.items():
    print(f"  {TARGET_SCHEMA}.{table}: {rows} rows")

In [0]:
# Validate all tables were created and show sample data
for table_name in TABLES.values():
    full_table = f"{TARGET_SCHEMA}.{table_name}"
    print(f"\n{'='*60}")
    print(f"Table: {full_table}")
    print(f"{'='*60}")
    df = spark.table(full_table)
    print(f"Schema:")
    df.printSchema()
    print(f"Sample (3 rows):")
    display(df.limit(3))